# Notebook 2 — Text-to-Image with Stable Diffusion

In this notebook we move from training a small model from scratch to using a
**pretrained latent diffusion model** (Stable Diffusion 1.5).

> **Requires:** GPU with ≥8 GB VRAM. Works on Colab T4.

In [ ]:

!pip install -q diffusers transformers accelerate peft safetensors


In [ ]:
import torch
import matplotlib.pyplot as plt
import numpy as np
from diffusers import StableDiffusionPipeline, DDIMScheduler, DDPMScheduler

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", device)

---
## 1. Stable Diffusion Architecture

Stable Diffusion is a **Latent Diffusion Model** (Rombach et al., CVPR 2022).
Instead of running diffusion in pixel space (can be large), it works in a compressed
**latent space** (see data/vis/stable_diffusion_architecture.html):

```
Text prompt
    │
    ▼
┌──────────┐     ┌──────────────┐     ┌──────────┐
│  CLIP    │ ───>│  U-Net       │ ───>│  VAE     │───>  Image
│  Text    │     │  (denoises   │     │  Decoder │     (512×512)
│  Encoder │     │   in latent  │     │          │
└──────────┘     │   space)     │     └──────────┘
                 └──────────────┘
                   64×64 latents
```

- **VAE Encoder/Decoder:** Compresses 512×512 images to 64×64 latents (8× reduction)
  and back.
- **U-Net:** Performs the diffusion (denoising) in latent space, conditioned on text
  embeddings.
- **CLIP Text Encoder:** Converts the text prompt into embeddings that guide the U-Net.
- **Classifier-Free Guidance (CFG):** At each step, the model predicts noise both with
  and without the text condition, then interpolates:
  `pred = pred_uncond + guidance_scale * (pred_cond - pred_uncond)`.

---
## 2. Load the Model and Generate Images

In [ ]:
model_id = "pt-sk/stable-diffusion-1.5"

pipe = StableDiffusionPipeline.from_pretrained(
    model_id,
    torch_dtype=torch.float16,
    safety_checker=None,  # disable for workshop speed
).to(device)

## DDIM vs DDPM

In [ ]:
# As before, ddim is faster but less precise
SAMPLERS = [
    {"sampler_id": "ddim", "num_steps": 30},
    {"sampler_id": "ddpm", "num_steps": pipe.scheduler.config.num_train_timesteps},
]

prompt = "a cat on a synthesizer in space, digital art, highly detailed"

fig, axes = plt.subplots(1, len(SAMPLERS), figsize=(6 * len(SAMPLERS), 6))

for ax, config in zip(axes, SAMPLERS):
    # Set the scheduler for this run
    if config["sampler_id"] == "ddim":
        pipe.scheduler = DDIMScheduler.from_config(pipe.scheduler.config)
    elif config["sampler_id"] == "ddpm":
        pipe.scheduler = DDPMScheduler.from_config(pipe.scheduler.config)

    image = pipe(
        prompt,
        num_inference_steps=config["num_steps"]-1,
        guidance_scale=7.5,
        generator=torch.Generator(device).manual_seed(42),
    ).images[0]

    ax.imshow(image)
    ax.set_title(f'{config["sampler_id"].upper()} ({config["num_steps"]} steps)', fontsize=12)
    ax.axis("off")

plt.suptitle(prompt, fontsize=11)
plt.tight_layout()
plt.show()


---
## 3. Exploring Key Parameters

Let's see how **guidance scale** and **number of steps** affect the output.

In [ ]:
# set scheduler as ddim for speed

pipe.scheduler = DDIMScheduler.from_config(pipe.scheduler.config)
NUM_INFERENCE_STEPS = 30

In [ ]:
# --- Effect of guidance_scale ---
prompt = "a beautiful sunset over mountains, oil painting"
scales = [1.0, 3.0, 7.5, 15.0]

fig, axes = plt.subplots(1, 4, figsize=(20, 5))
for ax, scale in zip(axes, scales):
    img = pipe(
        prompt,
        num_inference_steps=NUM_INFERENCE_STEPS,
        guidance_scale=scale,
        generator=torch.Generator(device).manual_seed(42),
    ).images[0]
    ax.imshow(img)
    ax.set_title(f"guidance = {scale}")
    ax.axis("off")
plt.suptitle(f"Effect of Classifier-Free Guidance Scale ({SAMPLER.upper()})", fontsize=14)
plt.tight_layout()
plt.show()

In [ ]:
# --- Effect of num_inference_steps ---

steps_list = [5, 10, 20, 50]

fig, axes = plt.subplots(1, 4, figsize=(20, 5))
for ax, steps in zip(axes, steps_list):
    img = pipe(
        prompt,
        num_inference_steps=steps,
        guidance_scale=7.5,
        generator=torch.Generator(device).manual_seed(42),
    ).images[0]
    ax.imshow(img)
    ax.set_title(f"steps = {steps}")
    ax.axis("off")
plt.suptitle(f"Effect of Number of Inference Steps ({SAMPLER.upper()})", fontsize=14)
plt.tight_layout()
plt.show()

---
## 4. Visualising the Denoising Process

We can intercept intermediate latents during sampling to see how the image emerges
from noise.

In [ ]:
prompt = "a futuristic city at night, neon lights, cyberpunk"

# We'll manually run the pipeline to capture intermediate steps
num_steps = NUM_INFERENCE_STEPS
pipe.scheduler.set_timesteps(num_steps, device=device)

# Encode prompt
text_input = pipe.tokenizer(
    prompt,
    return_tensors="pt",
    padding="max_length",
    max_length=pipe.tokenizer.model_max_length,
    truncation=True,
).to(device)

with torch.no_grad():
    text_emb = pipe.text_encoder(text_input.input_ids)[0]

# Unconditional embedding for CFG
uncond_input = pipe.tokenizer(
    [""],
    return_tensors="pt",
    padding="max_length",
    max_length=pipe.tokenizer.model_max_length,
).to(device)
with torch.no_grad():
    uncond_emb = pipe.text_encoder(uncond_input.input_ids)[0]

text_embeddings = torch.cat([uncond_emb, text_emb])

# Start from noise
generator = torch.Generator(device).manual_seed(42)
latents = torch.randn(
    1, 4, 64, 64, device=device, dtype=torch.float16, generator=generator
)
latents = latents * pipe.scheduler.init_noise_sigma

# Denoise and capture snapshots
snapshots = []
capture_at = [0, num_steps // 6, num_steps // 3, num_steps // 2,
              2 * num_steps // 3, num_steps - 1]  # evenly spaced captures

for i, t in enumerate(pipe.scheduler.timesteps):
    latent_input = torch.cat([latents] * 2)
    latent_input = pipe.scheduler.scale_model_input(latent_input, t)

    with torch.no_grad():
        noise_pred = pipe.unet(
            latent_input, t, encoder_hidden_states=text_embeddings
        ).sample

    noise_uncond, noise_cond = noise_pred.chunk(2)
    noise_pred = noise_uncond + 7.5 * (noise_cond - noise_uncond)

    latents = pipe.scheduler.step(noise_pred, t, latents).prev_sample

    if i in capture_at:
        with torch.no_grad():
            decoded = pipe.vae.decode(
                latents / pipe.vae.config.scaling_factor
            ).sample
        decoded = (decoded / 2 + 0.5).clamp(0, 1)
        snapshots.append((i, decoded[0].cpu().float().permute(1, 2, 0).numpy()))

# Plot
fig, axes = plt.subplots(1, len(snapshots), figsize=(20, 4))
for ax, (step, img) in zip(axes, snapshots):
    ax.imshow(img)
    ax.set_title(f"Step {step}")
    ax.axis("off")
plt.suptitle("Denoising process in latent space (decoded to pixels)", fontsize=14)
plt.tight_layout()
plt.show()